# 02 · Extract & Clean — Stage C
**Step 12: `extract_page`** turns one PDF page into structured lines, footnotes and metadata.

Rules come from Stage B (`01_inspect_profile.ipynb`):
- count and extract with **upright** chars only; the rotated side label is saved as `part`
- footer zone `top ≥ 780` → printed page number (dropped from text)
- footnote zone `650 ≤ top < 780` and `size ≤ 8.5` → moved to `footnotes`
- `size < 7.5` elsewhere → superscript, dropped
- text comes from `line["text"]` (ligatures appear twice in `line["chars"]`); fonts come from `line["chars"]`
- fewer than 200 upright chars → `needs_ocr` (step 15)

## 12.0 Setup

In [ ]:
import re, json
from collections import Counter

import pdfplumber
import pandas as pd

from src.paths import ROOT, PDF_PATH, PROFILE, DEC_PATH

In [ ]:


# Thresholds from Stage B
FOOTER_TOP     = 780    # at or below this top → footer (page number)
FN_TOP         = 650    # footnote zone starts here
FN_MAX_SIZE    = 8.5    # footnote font size ceiling
SUP_MAX_SIZE   = 7.5    # smaller than this (outside footnote zone) → superscript
MIN_REAL_CHARS = 200    # fewer upright chars → OCR page

# (font family, rounded size) → heading level. Anything else = body (Bold 11 is emphasis)
HEADING_RULES = {
    ("Bold", 30):     "L0",
    ("Bold", 14):     "L1",
    ("Semibold", 13): "L2",
    ("Semibold", 12): "L3",
}

DEC = json.loads(DEC_PATH.read_text(encoding="utf-8"))
print("manual_decisions keys:", list(DEC))

manual_decisions keys: ['footer_top', 'footnote_zone', 'footnote_size', 'superscript_max', 'min_chars', 'cover_pages', 'contents_pages', 'glossary_pages', 'form_pages', 'ocr_pages', 'diagram_text', 'oddities', 'char_replacements']


In [ ]:
page_map = pd.read_csv(PROFILE / "page_map.csv", dtype={"printed_page": str})
print(page_map.columns.tolist())
page_map.head()

['pdf_page', 'footer_text', 'pdf_label', 'printed_page', 'status']


,pdf_page,footer_text,pdf_label,printed_page,status
0,1,NaN,i,i,No Footer
1,2,NaN,ii,ii,No Footer
2,3,NaN,i,i,No Footer
3,4,NaN,ii,ii,No Footer
4,5,NaN,iii,iii,No Footer


In [ ]:
PDF_COL, PRINTED_COL = "pdf_page", "printed_page"

def to_label(v):
    """Printed page as text: '58', 'iv', or None if missing."""
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s.endswith(".0"):          # pandas sometimes reads 58 as 58.0
        s = s[:-2]
    return s.lower() or None

PAGE_MAP = {
    int(r[PDF_COL]): to_label(r[PRINTED_COL])
    for _, r in page_map.iterrows()
    if to_label(r[PRINTED_COL])
}
print(len(PAGE_MAP), "| PDF 78 →", PAGE_MAP.get(78), "| first five:", list(PAGE_MAP.items())[:5])

174 | PDF 78 → 58 | first five: [(1, 'i'), (2, 'ii'), (3, 'i'), (4, 'ii'), (5, 'iii')]


In [ ]:
print(page_map.columns.tolist())      # does the CSV say where each label came from?

def footer_text(page):
    chars = [c for c in page.chars if c.get("upright", True) and c["top"] >= FOOTER_TOP]
    chars.sort(key=lambda c: (round(c["top"]), c["x0"]))
    return "".join(c["text"] for c in chars).strip()

with pdfplumber.open(PDF_PATH) as pdf:
    for n in range(1, 23):
        page = pdf.pages[n - 1]
        n_up = sum(1 for c in page.chars if c.get("upright", True))
        print(f"PDF {n:>3} | footer read: {footer_text(page)!r:<8} | csv label: {PAGE_MAP.get(n)!r:<7} | upright chars: {n_up}")

['pdf_page', 'footer_text', 'pdf_label', 'printed_page', 'status']
PDF   1 | footer read: ''       | csv label: 'i'     | upright chars: 0
PDF   2 | footer read: ''       | csv label: 'ii'    | upright chars: 0
PDF   3 | footer read: ''       | csv label: 'i'     | upright chars: 153
PDF   4 | footer read: ''       | csv label: 'ii'    | upright chars: 0
PDF   5 | footer read: ''       | csv label: 'iii'   | upright chars: 0
PDF   6 | footer read: ''       | csv label: 'iv'    | upright chars: 0
PDF   7 | footer read: ''       | csv label: 'v'     | upright chars: 0
PDF   8 | footer read: ''       | csv label: 'vi'    | upright chars: 0
PDF   9 | footer read: ''       | csv label: 'vii'   | upright chars: 0
PDF  10 | footer read: ''       | csv label: 'viii'  | upright chars: 0
PDF  11 | footer read: ''       | csv label: 'ix'    | upright chars: 0
PDF  12 | footer read: ''       | csv label: 'x'     | upright chars: 0
PDF  13 | footer read: ''       | csv label: 'xi'    | upright char

In [ ]:
PDF_COL = "pdf_page"

# Use this for pages that must NOT show a printed label (fill in after checking the viewer)
LABEL_OVERRIDES = {1: None, 2: None}     # cover pages: label duplicates pages 3–4

PAGE_MAP, LABEL_SOURCE = {}, {}
for _, r in page_map.iterrows():
    n        = int(r[PDF_COL])
    footer   = to_label(r["footer_text"])
    printed  = to_label(r["printed_page"])
    file_lab = to_label(r["pdf_label"])

    if n in LABEL_OVERRIDES:
        label, source = LABEL_OVERRIDES[n], "override"
    elif footer and footer == printed:
        label, source = footer, "footer"          # read from the page itself
    elif file_lab:
        label, source = file_lab, "pdf_label"     # from the PDF file's built-in labels
    else:
        label, source = None, None

    PAGE_MAP[n], LABEL_SOURCE[n] = label, source

# Checks
print(pd.Series(LABEL_SOURCE).value_counts(dropna=False))
labels = pd.Series({k: v for k, v in PAGE_MAP.items() if v})
dupes = labels[labels.duplicated(keep=False)]
print("duplicate labels:", dupes.to_dict() or "none ✅")

footer       155
pdf_label     17
override       2
Name: count, dtype: int64
duplicate labels: none ✅


In [ ]:
page_map.loc[page_map["pdf_page"] <= 16, ["pdf_page", "footer_text", "pdf_label", "printed_page", "status"]]
print(page_map["status"].value_counts())

status
Matched      155
No Footer     19
Name: count, dtype: int64


## 12.1 `classify_char` — one decision per character
Order matters: side label → footer → footnote → superscript → keep.
A size-11 line sitting in the 650–780 zone stays in the body (only **small** text there is a footnote).

In [ ]:
def classify_char(obj):
    """Return where a pdfplumber object belongs: keep / side_label / footer / footnote / superscript."""
    if obj.get("object_type") != "char":
        return "keep"                      # lines, rects, curves — leave for tables (step 14)
    if not obj.get("upright", True):
        return "side_label"
    top, size = obj["top"], obj["size"]
    if top >= FOOTER_TOP:
        return "footer"
    if FN_TOP <= top < FOOTER_TOP and size <= FN_MAX_SIZE:
        return "footnote"
    if size < SUP_MAX_SIZE:
        return "superscript"
    return "keep"


def only(kind):
    """Filter for page.filter(): keep chars of one kind, keep all non-char objects."""
    def test(obj):
        if obj.get("object_type") != "char":
            return True
        return classify_char(obj) == kind
    return test

## 12.2 Side label → `part`
The label is rotated 90°. The char's `matrix[1]` (the *b* value) tells the direction:
- `b > 0` → text runs **bottom → top** (left pages) → read with `top` descending
- `b < 0` → text runs **top → bottom** (right pages) → read with `top` ascending

This is why sorting by `top` alone gave the reversed label on left pages.

In [ ]:
def read_side_label(page):
    chars = [c for c in page.chars if not c.get("upright", True) and c["text"].strip()]
    if not chars:
        return None
    direction = Counter(1 if c["matrix"][1] > 0 else -1 for c in chars).most_common(1)[0][0]
    reverse = direction > 0
    chars = sorted(chars, key=lambda c: c["top"], reverse=reverse)

    out = [chars[0]["text"]]
    for prev, c in zip(chars, chars[1:]):
        # empty distance between two letters, measured along the reading direction
        gap = (prev["top"] - c["bottom"]) if reverse else (c["top"] - prev["bottom"])
        if gap > 0.25 * prev["size"]:          # gap wider than about a quarter letter → word break
            out.append(" ")
        out.append(c["text"])
    return re.sub(r"\s+", " ", "".join(out)).strip() or None

## 12.3 Printed page number (footer) — falls back to `page_map.csv`

In [ ]:
ROMAN = re.compile(r"[ivxlc]+", re.I)

def read_printed_page(page, pdf_no):
    footer = [c for c in page.chars if c.get("upright", True) and c["top"] >= FOOTER_TOP]
    footer.sort(key=lambda c: (round(c["top"]), c["x0"]))
    text = "".join(c["text"] for c in footer).strip()

    m = re.search(r"\d+", text)
    if m:
        return m.group()                  # "58"
    if ROMAN.fullmatch(text):             # the whole footer is a numeral like "iv"
        return text.lower()
    return PAGE_MAP.get(pdf_no)           # fall back to Stage B's page map

## 12.4 Heading level from the line's dominant font
Check **Semibold before Bold**, because `"semibold"` also contains `"bold"`.

In [ ]:
def font_family(fontname):
    name = fontname.split("+")[-1].lower()     # drop the subset prefix, e.g. "ABCDEF+"
    if "semibold" in name:
        return "Semibold"
    if "bold" in name:
        return "Bold"
    return "Regular"


def heading_level(line):
    """Return (level, family, size) using the most common font among the line's visible chars."""
    chars = [c for c in line["chars"] if c["text"].strip()]
    if not chars:
        return "body", None, None
    (family, size), _ = Counter(
        (font_family(c["fontname"]), round(c["size"])) for c in chars
    ).most_common(1)[0]
    return HEADING_RULES.get((family, size), "body"), family, size

## 12.5 `extract_page`

In [ ]:
def extract_page(page, pdf_no):
    n_upright = sum(1 for c in page.chars if c.get("upright", True))

    result = {
        "pdf_page":     pdf_no,
        "printed_page": PAGE_MAP.get(pdf_no),
        "part":         None,
        "method":       "text",
        "n_upright":    n_upright,
        "lines":        [],
        "footnotes":    [],
    }

    # 1. Scanned page → OCR later (step 15)
    if n_upright < MIN_REAL_CHARS:
        result["method"] = "needs_ocr"
        return result

    # 2–3. Metadata
    result["part"]         = read_side_label(page)
    result["printed_page"] = read_printed_page(page, pdf_no)

    # 4. Footnotes (moved out of the body)
    fn_page = page.filter(only("footnote"))
    result["footnotes"] = [l["text"].strip() for l in fn_page.extract_text_lines() if l["text"].strip()]

    # 5. Body — filtered BEFORE line building, so superscripts / labels / footer never enter the text
    # TODO step 14: also exclude table bounding boxes here
    body_page = page.filter(only("keep"))
    for l in body_page.extract_text_lines(return_chars=True):
        text = l["text"].strip()          # from line["text"], not chars (ligature duplicates)
        if not text:
            continue
        level, family, size = heading_level(l)
        result["lines"].append({
            "text":  text,
            "level": level,
            "font":  family,
            "size":  size,
            "top":   round(l["top"], 1),
        })

    return result

## 12.6 Look at single pages

In [ ]:
def show(pdf_no, max_lines=25):
    with pdfplumber.open(PDF_PATH) as pdf:
        r = extract_page(pdf.pages[pdf_no - 1], pdf_no)
    print(f"PDF {r['pdf_page']} | printed {r['printed_page']} | part {r['part']!r} | "
          f"{r['method']} | upright {r['n_upright']} | lines {len(r['lines'])} | footnotes {len(r['footnotes'])}")
    for l in r["lines"][:max_lines]:
        tag = l["level"] if l["level"] != "body" else f"   {l['font'][:4]} {l['size']}"
        print(f"  {tag:<12} {l['text'][:95]}")
    for f in r["footnotes"]:
        print("  [fn]", f[:95])
    return r

_ = show(21)   # chapter start → one L0, then L1/L2

PDF 21 | printed 1 | part 'INTRODUCTION' | text | upright 2303 | lines 29 | footnotes 2
  L1           Chapter
     Regu 60   1
  L0           Introduction
     Regu 11   The National Health Mission is a flagship health sector reform initiative started in 2005 as
     Regu 11   National Rural Health Mission (NRHM) and was later renamed National Health Mission (NHM)
     Regu 11   when National Urban Health Mission (NUHM) was conceptualized as a sub mission under NHM.
     Regu 11   The vision of NHM is: “Attainment of Universal Access to Equitable, Affordable and Quality
     Regu 11   health care services, accountable and responsive to people’s needs, with effective inter-sector
     Regu 11   convergent action to address the wider social determinants of health”
     Regu 11   Over the years, the NHM has initiated multiple health system reforms to strengthen primary
     Regu 11   and secondary care. NHM is both flexible and dynamic and is intended to guide States towards
     Regu 11

In [ ]:
with pdfplumber.open(PDF_PATH) as pdf:
    p = pdf.pages[20]
    for c in p.chars:
        if c["text"] == "1" and c["top"] < FN_TOP:
            near = sorted([d for d in p.chars if abs(d["top"] - c["top"]) < 6 and abs(d["x0"] - c["x0"]) < 60],
                          key=lambda d: d["x0"])
            print(f"size {c['size']:.2f} | top {c['top']:.1f} | {classify_char(c):<11} | …{''.join(d['text'] for d in near)}…")

size 6.41 | top 384.9 | superscript | …ts of health1” …
size 11.00 | top 491.9 | keep        | …cal barriers.1 The NHM …
size 6.41 | top 542.9 | superscript | …Core Values1  driving the …
size 60.00 | top 77.2 | keep        | …1…


In [ ]:
with pdfplumber.open(PDF_PATH) as pdf:
    p = pdf.pages[20]                     # PDF 21
    for c in p.chars:
        if c["text"] == "1" and c["top"] < FN_TOP:
            print(f"size {c['size']:.2f} | top {c['top']:.1f} | font {c['fontname']}")

size 6.41 | top 384.9 | font EUUBGP+ProximaNovaA-Regular
size 11.00 | top 491.9 | font EUUBGP+ProximaNovaA-Regular
size 6.41 | top 542.9 | font EUUBGP+ProximaNovaA-Regular
size 60.00 | top 77.2 | font VHPGYC+Helvetica-Condensed-Light


In [ ]:
_ = show(4)

PDF 4 | printed ii | part None | needs_ocr | upright 0 | lines 0 | footnotes 0


In [ ]:
_ = show(78)   # printed_page should be 58

PDF 78 | printed 58 | part 'SUGGESTIVE PROCEDURAL FORMATS' | text | upright 1541 | lines 30 | footnotes 2
  L1           Induction training Schedule of M&E Officer
     Bold 11   Day Thematic area Description
     Regu 11   Day 1- Day 8 Initial Orientation
     Regu 11   Day 1 Introduction to National Health Objective and strategies, structure and
     Regu 11   Mission function of NHM
     Regu 11   Day 2 Orientation of Duties and Role of the incumbent, overall
     Regu 11   Responsibilities responsibilities specific to monitoring and
     Regu 11   evaluation
     Regu 11   Day 3 Monitoring and evaluation Constitution of M&E Division, Basic duties
     Regu 11   system of each team member. Details of all
     Regu 11   process related to Resource allocation and
     Regu 11   utilization
     Regu 11   Day 4 Introduction to the partner Sessions on the role and involvement
     Regu 11   departments of water & sanitation, women & child
     Regu 11   development
     Regu 11   Day 5 

In [ ]:
_ = show(32)   # flowchart page — note what comes out; replaced by manual text later

PDF 32 | printed 12 | part 'RECRUITMENT' | text | upright 2606 | lines 38 | footnotes 3
     Regu 11   support staff (such as cooks, cleaners, etc., mostly hired through an agency). Presence of a
     Regu 11   medical college or a nursing school in the district makes it easier for the district to recruit
     Regu 11   passing out students.
     Regu 11   State may determine as to which posts could be recruited by the districts, based on their capac
     Regu 11   to undertake the recruitment processes. The HR cell should support and monitor the process
     Regu 11   even if the recruitment is delegated to the district. While the responsibility can be delegated
     Regu 11   accountability remains with the State.
     Regu 11   To enable speedy and professional recruitment, the State or district may use the empanelled
     Regu 11   Recruitment agencies of MoHFW or empanel their own agencies . These agencies would
     Regu 11   carry out the recruitment up to the interview as per t

In [ ]:
offsets = {n: n - int(l) for n, l in PAGE_MAP.items() if l and l.isdigit()}
pd.Series(offsets).value_counts()

20    154
Name: count, dtype: int64

## 12.7 Run all pages and check

In [ ]:
rows, RESULTS = [], {}
with pdfplumber.open(PDF_PATH) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        r = extract_page(page, i)
        RESULTS[i] = r
        rows.append({
            "pdf_page":     i,
            "method":       r["method"],
            "printed_page": r["printed_page"],
            "map_printed":  PAGE_MAP.get(i),
            "part":         r["part"],
            "n_lines":      len(r["lines"]),
            "n_headings":   sum(l["level"] != "body" for l in r["lines"]),
            "n_footnotes":  len(r["footnotes"]),
        })
        page.flush_cache()

summary = pd.DataFrame(rows)
summary["method"].value_counts()      # expect 118 text + OCR pages

method
text         118
needs_ocr     56
Name: count, dtype: int64

In [ ]:
# 1) Footer number vs Stage B page_map → should be empty
mismatch = summary.dropna(subset=["map_printed"])
mismatch = mismatch[mismatch["printed_page"] != mismatch["map_printed"]]
print("printed-page mismatches:", len(mismatch))
display(mismatch)

# 2) Side labels found — every value should read forwards (no "TNEMTIURCER")
print(summary["part"].value_counts(dropna=False))

printed-page mismatches: 0


,pdf_page,method,printed_page,map_printed,part,n_lines,n_headings,n_footnotes


part
SUGGESTIVE PROCEDURAL FORMATS             59
NaN                                       56
LEARNINGS FROM OTHER STATES               24
RECRUITMENT                                6
EMPLOYEE MANAGEMENT                        6
CONTENTS                                   4
HRH POLICY, PRINCIPLES AND STRATEGIES      3
MODES OF ENGAGEMENT UNDER NHM              3
CAPACITY BUILDING/CAPACITY DEVELOPMENT     3
LIST OF ABBREVIATIONS                      2
INTRODUCTION                               2
HUMAN RESOURCE FOR HEALTH UNDER NHM        2
APPOINTMENT AND ONBOARDING                 2
CODE OF CONDUCT                            2
Name: count, dtype: int64


In [ ]:
# 3) Pick a left page with a side label and a page with footnotes, then look at them
left_with_label = summary[(summary["part"].notna()) & (summary["pdf_page"] % 2 == 0)]["pdf_page"].head(3).tolist()
with_footnotes  = summary[summary["n_footnotes"] > 0]["pdf_page"].head(3).tolist()
print("left pages with label:", left_with_label, "| pages with footnotes:", with_footnotes)

if with_footnotes:
    _ = show(with_footnotes[0])

left pages with label: [16, 18, 20] | pages with footnotes: [19, 21, 23]
PDF 19 | printed xvii | part 'CONTENTS' | text | upright 1666 | lines 40 | footnotes 3
     Bold 11   Chapter 9. Code of Conduct 28-29
     Regu 11   9.1. Contents of Code of Conduct Policy 28
     Regu 11   9.2. Breach of Code of Conduct 29
     Regu 11   9.3. Anti-Harassment Committee 29
     Bold 11   Annexure I. Suggestive Procedural Formats 30-88
     Regu 11   Deputation Principles and Process 30
     Regu 11   Process of Designing Job Description/ToR 32
     Regu 11   Sample ToRs/ Roles and Responsibilities 34
     Regu 11   Stages of Selection 51
     Regu 11   Contents of Offer Letter 53
     Regu 11   Contents of the Contract 54
     Regu 11   Induction Training Schedule of M&E Officer 58
     Regu 11   Training Needs Assessment 59
     Regu 11   Process of Performance Appraisal 60
     Regu 11   Expenses Policy 68
     Regu 11   Sample Travel Approval Form 70
     Regu 11   Sample Requisition for Advanc

In [ ]:
# 4) Superscript leak check: body lines ending in a digit glued to a word, e.g. "recruitment1"
leaks = [(p, l["text"]) for p, r in RESULTS.items() for l in r["lines"]
         if re.search(r"[a-z]\d{1,2}$", l["text"])]
print(len(leaks), "possible leaks")
leaks[:10]

1 possible leaks


[(146, 'Learnings from Other States13')]

In [ ]:
# 5) Heading levels found across the chapter body (PDF 21–49)
pd.DataFrame(
    [{"pdf_page": p, **l} for p, r in RESULTS.items() if 21 <= p <= 49 for l in r["lines"] if l["level"] != "body"]
)[["pdf_page", "level", "font", "size", "text"]].head(40)

,pdf_page,level,font,size,text
0,21,L1,Bold,14,Chapter
1,21,L0,Bold,30,Introduction
2,22,L1,Bold,14,1.1. Purpose of the Guidelines
3,23,L1,Bold,14,Chapter
4,23,L0,Bold,30,Human Resource for Health
5,23,L0,Bold,30,under NHM
6,23,L2,Semibold,13,2.1.1. Service Delivery
7,23,L2,Semibold,13,2.1.2. Programme Management (PM)2
8,24,L2,Semibold,13,2.1.3. Support Service
9,24,L2,Semibold,13,2.1.4. Other HR


In [ ]:
_ = show(78)

PDF 78 | printed 58 | part 'SUGGESTIVE PROCEDURAL FORMATS' | text | upright 1541 | lines 30 | footnotes 2
  L1           Induction training Schedule of M&E Officer
     Bold 11   Day Thematic area Description
     Regu 11   Day 1- Day 8 Initial Orientation
     Regu 11   Day 1 Introduction to National Health Objective and strategies, structure and
     Regu 11   Mission function of NHM
     Regu 11   Day 2 Orientation of Duties and Role of the incumbent, overall
     Regu 11   Responsibilities responsibilities specific to monitoring and
     Regu 11   evaluation
     Regu 11   Day 3 Monitoring and evaluation Constitution of M&E Division, Basic duties
     Regu 11   system of each team member. Details of all
     Regu 11   process related to Resource allocation and
     Regu 11   utilization
     Regu 11   Day 4 Introduction to the partner Sessions on the role and involvement
     Regu 11   departments of water & sanitation, women & child
     Regu 11   development
     Regu 11   Day 5 

In [ ]:
# out = ROOT / "data" / "interim" / "pages_raw.jsonl"
# out.parent.mkdir(parents=True, exist_ok=True)
# with out.open("w", encoding="utf-8") as f:
#     for n in sorted(RESULTS):
#         f.write(json.dumps(RESULTS[n], ensure_ascii=False) + "\n")
# print("saved", len(RESULTS), "pages →", out)


#build path.py -->all path exist here
#so import link from there
#making pipeline of link 

from src.paths import PAGES_RAW

PAGES_RAW.parent.mkdir(parents=True, exist_ok=True)
with PAGES_RAW.open("w", encoding="utf-8") as f:
    for n in sorted(RESULTS):
        f.write(json.dumps(RESULTS[n], ensure_ascii=False) + "\n")
print("saved", len(RESULTS), "pages →", PAGES_RAW.relative_to(ROOT))

saved 174 pages → data\interim\pages_raw.jsonl


## 13.0 Setup

In [ ]:
import re, json, copy
from collections import Counter
import pandas as pd
from src.paths import ROOT

from src.paths import PAGES_RAW, PAGES_CLEAN

RAW_PATH   = PAGES_RAW      # same variable names as before, so the other cells keep working
CLEAN_PATH = PAGES_CLEAN

PAGES = [json.loads(line) for line in RAW_PATH.open(encoding="utf-8")]
print(len(PAGES), "pages loaded")

FORM_PAGES     = set(range(84, 88)) | set(range(90, 96)) | set(range(105, 109))
HEADING_LEVELS = {"L0", "L1", "L2", "L3"}

CHANGES = []    # every change is logged here, so we can review what the rules did
def log(page, rule, before, after):
    CHANGES.append({"pdf_page": page, "rule": rule, "before": before, "after": after})

174 pages loaded


## 13.A Headings

In [ ]:
SECTION_NUM = re.compile(r"^\d+(\.\d+)*\.?\s")                 # "4.2. ..." starts a new heading
CAPTION     = re.compile(r"^(Exhibit|Table|Figure)\s*\d+", re.I)

def is_chapter_word(l):
    return l["level"] == "L1" and l["text"].strip().lower() == "chapter"

def is_big_number(l):
    return l["text"].strip().isdigit() and (l.get("size") or 0) >= 30

def merge_chapter_openers(lines, pdf_page):
    """'Chapter' + '1' (size 60) + title L0 line(s)  ->  one L0 'Chapter 1: Title'"""
    out, i = [], 0
    while i < len(lines):
        l = lines[i]
        if is_chapter_word(l):
            num_idx = next((k for k in (i + 1, i - 1)
                            if 0 <= k < len(lines) and is_big_number(lines[k])), None)
            if num_idx is not None:
                j = max(i, num_idx) + 1
                title = []
                while j < len(lines) and lines[j]["level"] == "L0":
                    title.append(lines[j]["text"].strip())
                    j += 1
                if title:
                    if num_idx == i - 1 and out and out[-1] is lines[i - 1]:
                        out.pop()                       # number came before "Chapter"
                    new_text = f"Chapter {lines[num_idx]['text'].strip()}: {' '.join(title)}"
                    out.append({**l, "text": new_text, "level": "L0"})
                    log(pdf_page, "chapter_merge", "Chapter | " + " | ".join(title), new_text)
                    i = j
                    continue
            print(f"⚠️ PDF {pdf_page}: 'Chapter' found but number/title missing — left unchanged")
        out.append(l)
        i += 1
    return out

def join_split_headings(lines, pdf_page):
    """Two neighbouring heading lines of the same level, close together -> one heading"""
    out, last_top = [], None
    for l in lines:
        prev = out[-1] if out else None
        if (prev is not None
                and l["level"] in HEADING_LEVELS
                and l["level"] == prev["level"]
                and not SECTION_NUM.match(l["text"])
                and not l["text"].startswith("Chapter ")
                and 0 < l["top"] - last_top <= 2 * (l.get("size") or 12)):
            joined = prev["text"].rstrip() + " " + l["text"].strip()
            log(pdf_page, "heading_join", prev["text"] + " | " + l["text"], joined)
            out[-1] = {**prev, "text": joined}
        else:
            out.append(l)
        last_top = l["top"]
    return out

def tag_captions(lines):
    for l in lines:
        if l["level"] in HEADING_LEVELS and CAPTION.match(l["text"].strip()):
            l["level"] = "caption"
    return lines

def fix_headings(lines, pdf_page):
    lines = merge_chapter_openers(lines, pdf_page)
    lines = join_split_headings(lines, pdf_page)
    return tag_captions(lines)

## 13.B Footnotes

In [ ]:
FN_START   = re.compile(r"^[*†]?(\d{1,3})(?:\s+\S|$)")   # "12 Text" · "*7 Text" · "9" alone
ORDINAL    = {"st", "nd", "rd", "th"}                     # raised letters of 2nd, 3rd... not footnotes
URL_AT_END = re.compile(r"(https?://|www\.)\S*$")
URL_DONE   = re.compile(r"\.(pdf|html?|aspx?|php|docx?|xlsx?)$", re.I)
URL_PIECE  = re.compile(r"^[\w\-./%?=&#~]+$")

def continues_url(prev_text, line):
    first = line.split()[0]
    return (URL_AT_END.search(prev_text) is not None
            and URL_DONE.search(prev_text) is None
            and URL_PIECE.match(first) is not None)

def fix_footnotes(footnotes, pdf_page):
    notes = []
    for raw in footnotes:
        line = raw.strip()
        if not line:
            continue

        # 1. raised letters of "2nd", "3rd" -> not a footnote
        if line.lower() in ORDINAL:
            log(pdf_page, "ordinal_dropped", line, "")
            continue

        m = FN_START.match(line)

        # 2. "9" on its own line, then "9 Complaints..." -> keep one
        if m and notes and notes[-1]["num"] == int(m.group(1)) and notes[-1]["text"] == m.group(1):
            notes[-1]["text"] = line.lstrip("*†")
            continue

        # 3. a new footnote starts (its number is higher than the previous one)
        if m and (not notes or notes[-1]["num"] is None or int(m.group(1)) > notes[-1]["num"]):
            notes.append({"num": int(m.group(1)), "text": line.lstrip("*†")})

        # 4. otherwise: continuation of the previous footnote
        elif notes:
            sep = "" if continues_url(notes[-1]["text"], line) else " "
            notes[-1]["text"] += sep + line

        # 5. text with no footnote before it
        else:
            print(f"⚠️ PDF {pdf_page}: footnote line without a number: {line[:60]!r}")
            notes.append({"num": None, "text": line})

    fn_numbers = {n["num"] for n in notes if n["num"] is not None}
    return notes, fn_numbers

## 13.C Body lines

In [ ]:
MARKER = re.compile(
    r"(?<=[A-Za-z\)\]”’\"'])(\d{1,2})(?=[\s,;:.”’\"')]|$)"   # States13 · (PM)2
    r"|(?<=[A-Za-z]\.)(\d{1,2})(?=\s|$)"                      # barriers.1 The
)
FILL_IN = re.compile(r"_{3,}|[.…]{4,}|…{2,}")

def drop_markers(text, fn_numbers):
    def repl(m):
        n = m.group(1) or m.group(2)
        word_start  = text.rfind(" ", 0, m.start()) + 1
        word_before = text[word_start:m.start()]
        if any(ch.isdigit() for ch in word_before):    # codes like H1N1: not a marker
            return m.group(0)
        return "" if int(n) in fn_numbers else m.group(0)
    return MARKER.sub(repl, text)


REAL_NUMBER_WORDS = re.compile(r"(Annexure|Part|Chapter|Table|Exhibit|Figure|Form|Section|Phase|Day)\s\d{1,2}$", re.I)

def drop_heading_end_marker(text, fn_numbers):
    """'...Bihar 14' -> '...Bihar' when 14 is a footnote on this page (headings only)"""
    if REAL_NUMBER_WORDS.search(text):
        return text
    return re.sub(r"\s(\d{1,2})$",
                  lambda m: "" if int(m.group(1)) in fn_numbers else m.group(0), text)


def fix_body(lines, fn_numbers, pdf_page):
    out = []
    for l in lines:
        t = original = l["text"]

        if fn_numbers:                                             # C1 typed markers
            t = drop_markers(t, fn_numbers)
            if l["level"] in HEADING_LEVELS:
                t = drop_heading_end_marker(t, fn_numbers)
            if t != original:
                log(pdf_page, "typed_marker", original, t)

        if t.startswith("y "):                                     # C2 Wingdings bullet
            t = "- " + t[2:]

        t = FILL_IN.sub("" if pdf_page in FORM_PAGES else "[__]", t)   # C4 fill-ins
        t = re.sub(r"\s+([.,;:])", r"\1", t)                       # C3 space before punctuation
        t = re.sub(r"\s{2,}", " ", t).strip()                      # C5 tidy spaces

        if pdf_page == 90 and t.startswith("Air tickets booked by myself"):   # C6 manual
            log(pdf_page, "manual_drop", original, "")
            continue
        if t:
            out.append({**l, "text": t})
    return out

## 13.D Characters

In [ ]:
CHAR_MAP = {"’": "'", "‘": "'", "“": '"', "”": '"', "–": "-", "—": "-",
            "√": "Yes", "\u00a0": " ", "\u200b": ""}

def normalize_chars(text):
    if not text:
        return text
    text = re.sub(r"^\s*•\s*", "- ", text)       # bullet at line start
    text = text.replace("•", "-")
    for old, new in CHAR_MAP.items():
        text = text.replace(old, new)
    return re.sub(r"\s{2,}", " ", text).strip()

## 13.E Run all pages

In [ ]:
def clean_page(page):
    page = copy.deepcopy(page)                   # never change PAGES, so before/after can be compared
    if page["method"] != "text":
        return page                              # OCR pages pass through untouched
    n = page["pdf_page"]
    notes, fn_numbers = fix_footnotes(page["footnotes"], n)   # B first: C needs the numbers
    lines = fix_headings(page["lines"], n)                    # A
    lines = fix_body(lines, fn_numbers, n)                    # C
    for l in lines:                                           # D on everything
        l["text"] = normalize_chars(l["text"])
    for note in notes:
        note["text"] = normalize_chars(note["text"])
    page["part"]      = normalize_chars(page["part"])
    page["lines"]     = [l for l in lines if l["text"]]
    page["footnotes"] = notes
    return page

CHANGES.clear()
CLEAN = [clean_page(p) for p in PAGES]
pd.DataFrame(CHANGES)["rule"].value_counts()

rule
heading_join       19
chapter_merge       9
ordinal_dropped     4
typed_marker        4
Name: count, dtype: int64

## 13.F Checks

In [ ]:
print("1) pages:", len(CLEAN))

chapters = [(p["pdf_page"], l["text"]) for p in CLEAN for l in p["lines"]
            if re.match(r"^Chapter \d+: ", l["text"])]
print("2) chapters:", len(chapters))
for c in chapters: print("   ", c)

LEAK  = re.compile(r"[a-z\)]\d{1,2}$|[a-z]\.\d{1,2}\s")
leaks = [(p["pdf_page"], l["text"]) for p in CLEAN for l in p["lines"] if LEAK.search(l["text"])]
print("3) possible leaks:", len(leaks), leaks[:10])

print("4) 'y ' bullets left:", sum(l["text"].startswith("y ") for p in CLEAN for l in p["lines"]))

left = Counter(ch for p in CLEAN for l in p["lines"] for ch in l["text"] if ord(ch) > 127)
print("5) non-ASCII characters left:", left.most_common(15))

def n_chars(pages): return sum(len(l["text"]) for p in pages for l in p["lines"])
b, a = n_chars(PAGES), n_chars(CLEAN)
print(f"6) body characters: {b:,} -> {a:,}  ({(b - a) / b:+.2%})")

1) pages: 174
2) chapters: 9
    (21, 'Chapter 1: Introduction')
    (23, 'Chapter 2: Human Resource for Health under NHM')
    (25, 'Chapter 3: HRH Policy, Principles and Strategies')
    (28, 'Chapter 4: Modes of Engagement under NHM')
    (31, 'Chapter 5: Recruitment')
    (37, 'Chapter 6: Appointment and Onboarding')
    (39, 'Chapter 7: Capacity Building/Capacity Development')
    (42, 'Chapter 8: Employee Management')
    (48, 'Chapter 9: Code of Conduct')
3) possible leaks: 0 []
4) 'y ' bullets left: 0
5) non-ASCII characters left: []
6) body characters: 229,191 -> 227,347  (+0.80%)


## 13.G Before / after for one page

In [ ]:
def compare(pdf_no, n=15):
    before = next(p for p in PAGES if p["pdf_page"] == pdf_no)
    after  = next(p for p in CLEAN if p["pdf_page"] == pdf_no)
    print(f"── PDF {pdf_no} BEFORE ──")
    for l in before["lines"][:n]: print(f"  {l['level']:<8}{l['text'][:90]}")
    for f in before["footnotes"]:  print("  [fn]", f[:90])
    print(f"── PDF {pdf_no} AFTER ──")
    for l in after["lines"][:n]:  print(f"  {l['level']:<8}{l['text'][:90]}")
    for f in after["footnotes"]:   print(f"  [fn {f['num']}]", f["text"][:90])

compare(21)

── PDF 21 BEFORE ──
  L1      Chapter
  body    1
  L0      Introduction
  body    The National Health Mission is a flagship health sector reform initiative started in 2005 
  body    National Rural Health Mission (NRHM) and was later renamed National Health Mission (NHM)
  body    when National Urban Health Mission (NUHM) was conceptualized as a sub mission under NHM.
  body    The vision of NHM is: “Attainment of Universal Access to Equitable, Affordable and Quality
  body    health care services, accountable and responsive to people’s needs, with effective inter-s
  body    convergent action to address the wider social determinants of health”
  body    Over the years, the NHM has initiated multiple health system reforms to strengthen primary
  body    and secondary care. NHM is both flexible and dynamic and is intended to guide States towar
  body    ensuring the achievement of universal access to health care through strengthening of healt
  body    systems, institutions, and capabi

## 13. Manual testing the output

In [ ]:
pd.set_option("display.max_colwidth", 120)
chg = pd.DataFrame(CHANGES)
chg[chg["rule"].isin(["heading_join", "typed_marker"])][["pdf_page", "rule", "after"]]

,pdf_page,rule,after
4,21,typed_marker,"gender, poverty, caste, other forms of social exclusion and geographical barriers. The NHM"
6,23,typed_marker,2.1.2. Programme Management (PM)
14,62,heading_join,Duties & Responsibilities related to Critical Care Unit (I.C.U./ Burn Unit/ H.D.U./ S.N.C.U. unit)
15,72,heading_join,Suggestive Principles for Calculation of Weightage for Education and Experience
16,75,heading_join,AGREEMENT OF CONTRACT APPOINTMENT UNDER STATE / DISTRICT HEALTH SOCIETY - ***
18,101,heading_join,Table: Example of determining entry level salary for the position of consultant based on experience and qualification
19,146,typed_marker,Learnings from Other States
20,147,heading_join,Promoting Better Integration of Health Information Systems: Best
21,147,heading_join,Promoting Better Integration of Health Information Systems: Best Practices
22,150,heading_join,Bringing Accountability in Health: The Madhya Pradesh


In [ ]:
for n in (35, 46, 47):
    page = next(p for p in CLEAN if p["pdf_page"] == n)
    for f in page["footnotes"]:
        print(n, f["num"], f["text"][:80])

35 7 7 Only suggestive, based on expert opinion and large-scale recruitments
46 9 9 Complaints pertaining to Sexual Harassment shall be referred to Anti-Sexual Ha
47 10 10 Refer: eHRMS Manav Sampada Microsoft Word - ManavsampadaHRMS Write up.docx (e


In [ ]:
for n in (46, 47, 164):
    page = next(p for p in CLEAN if p["pdf_page"] == n)
    print(n, [f["num"] for f in page["footnotes"]], "|", page["footnotes"][0]["text"][:60] if page["footnotes"] else "")
print([l["text"] for l in next(p for p in CLEAN if p["pdf_page"] == 164)["lines"] if "AMANAT" in l["text"]])

46 [9] | 9 Complaints pertaining to Sexual Harassment shall be referr
47 [10] | 10 Refer: eHRMS Manav Sampada Microsoft Word - ManavsampadaH
164 [14] | 14 Since 2018, the programme has been remodeled as AMANAT Jy
['AMANAT - Nurse Mobile Mentoring Programme - Bihar']


## 13.T Automated tests

In [ ]:
# ============================================================
# 13.T  Automated tests for steps 12 + 13
# ============================================================
TESTS = []

def check(group, name, get_actual, expected, desc=None):
    """Run one test. `expected` is a value, or a function returning True/False (then give `desc`)."""
    try:
        actual = get_actual()
        ok = expected(actual) if callable(expected) else actual == expected
    except Exception as e:
        actual, ok = f"ERROR {type(e).__name__}: {e}", False
    TESTS.append({"group": group, "test": name, "result": "PASS" if ok else "FAIL",
                  "expected": desc if callable(expected) else expected, "actual": actual})

def page(n, data):
    return next(p for p in data if p["pdf_page"] == n)

def all_lines(data):
    return [(p["pdf_page"], l) for p in data for l in p["lines"]]

EXPECTED_CHAPTERS = [
    "Chapter 1: Introduction",
    "Chapter 2: Human Resource for Health under NHM",
    "Chapter 3: HRH Policy, Principles and Strategies",
    "Chapter 4: Modes of Engagement under NHM",
    "Chapter 5: Recruitment",
    "Chapter 6: Appointment and Onboarding",
    "Chapter 7: Capacity Building/Capacity Development",
    "Chapter 8: Employee Management",
    "Chapter 9: Code of Conduct",
]
EXPECTED_PARTS = {
    "SUGGESTIVE PROCEDURAL FORMATS", "LEARNINGS FROM OTHER STATES", "RECRUITMENT",
    "EMPLOYEE MANAGEMENT", "CONTENTS", "HRH POLICY, PRINCIPLES AND STRATEGIES",
    "MODES OF ENGAGEMENT UNDER NHM", "CAPACITY BUILDING/CAPACITY DEVELOPMENT",
    "LIST OF ABBREVIATIONS", "INTRODUCTION", "HUMAN RESOURCE FOR HEALTH UNDER NHM",
    "APPOINTMENT AND ONBOARDING", "CODE OF CONDUCT",
}

# ---------- Step 12: extraction (uses PAGES = pages_raw.jsonl) ----------
G = "12 extract"
check(G, "page count", lambda: len(PAGES), 174)
check(G, "text pages", lambda: sum(p["method"] == "text" for p in PAGES), 118)
check(G, "needs_ocr pages", lambda: sum(p["method"] == "needs_ocr" for p in PAGES), 56)
check(G, "PDF 110 is needs_ocr", lambda: page(110, PAGES)["method"], "needs_ocr")
check(G, "PDF 21 printed page", lambda: page(21, PAGES)["printed_page"], "1")
check(G, "PDF 78 printed page", lambda: page(78, PAGES)["printed_page"], "58")
check(G, "offset printed = PDF - 20 (all numbered pages)",
      lambda: sorted({p["pdf_page"] - int(p["printed_page"]) for p in PAGES
                      if p["printed_page"] and str(p["printed_page"]).isdigit()}), [20])
check(G, "numbered pages count",
      lambda: sum(1 for p in PAGES if p["printed_page"] and str(p["printed_page"]).isdigit()), 154)
check(G, "printed page = page_map (mismatches)",
      lambda: sum(1 for p in PAGES if PAGE_MAP.get(p["pdf_page"]) is not None
                  and p["printed_page"] != PAGE_MAP[p["pdf_page"]]), 0)
check(G, "label sources",
      lambda: dict(Counter(v for v in LABEL_SOURCE.values() if v)),
      {"footer": 155, "pdf_label": 17, "override": 2})
check(G, "no duplicate page labels",
      lambda: len([v for v in PAGE_MAP.values() if v]) - len({v for v in PAGE_MAP.values() if v}), 0)
check(G, "PDF 21 side label", lambda: page(21, PAGES)["part"], "INTRODUCTION")
check(G, "PDF 78 side label (spaces kept)", lambda: page(78, PAGES)["part"], "SUGGESTIVE PROCEDURAL FORMATS")
check(G, "all side labels readable (known set)",
      lambda: sorted({p["part"] for p in PAGES if p["part"]} - EXPECTED_PARTS), [])
check(G, "heading levels L0-L3 all found",
      lambda: sorted({l["level"] for _, l in all_lines(PAGES)} & {"L0", "L1", "L2", "L3"}),
      ["L0", "L1", "L2", "L3"])

# ---------- Step 13: cleaning (uses CLEAN + CHANGES) ----------
G = "13 clean"
LEAK = re.compile(r"[a-z\)]\d{1,2}$|[a-z]\.\d{1,2}\s")
check(G, "page count", lambda: len(CLEAN), 174)
check(G, "chapters match TOC",
      lambda: [l["text"] for _, l in all_lines(CLEAN) if re.match(r"^Chapter \d+: ", l["text"])],
      EXPECTED_CHAPTERS)
check(G, "no marker leaks", lambda: [(n, l["text"]) for n, l in all_lines(CLEAN) if LEAK.search(l["text"])], [])
check(G, "no 'y ' bullets left", lambda: sum(l["text"].startswith("y ") for _, l in all_lines(CLEAN)), 0)
check(G, "no non-ASCII characters left",
      lambda: sorted({ch for _, l in all_lines(CLEAN) for ch in l["text"] if ord(ch) > 127}
                     | {ch for p in CLEAN for f in p["footnotes"] for ch in f["text"] if ord(ch) > 127}), [])
check(G, "no empty lines", lambda: sum(not l["text"].strip() for _, l in all_lines(CLEAN)), 0)
check(G, "characters changed < 1%",
      lambda: round(abs(sum(len(l["text"]) for _, l in all_lines(PAGES))
                        - sum(len(l["text"]) for _, l in all_lines(CLEAN)))
                    / sum(len(l["text"]) for _, l in all_lines(PAGES)) * 100, 2),
      lambda v: v < 1, "< 1 (%)")
check(G, "typed markers removed", lambda: sum(c["rule"] == "typed_marker" for c in CHANGES), 4)
check(G, "every footnote has a number",
      lambda: [(p["pdf_page"], f["text"][:40]) for p in CLEAN for f in p["footnotes"] if f["num"] is None], [])
check(G, "PDF 21 marker removed", lambda: any("barriers. The NHM" in l["text"] for l in page(21, CLEAN)["lines"]), True)
check(G, "PDF 46 footnote: one 9", lambda: page(46, CLEAN)["footnotes"][0]["text"][:12], "9 Complaints")
check(G, "PDF 47 footnote: one 10", lambda: page(47, CLEAN)["footnotes"][0]["text"][:8], "10 Refer")
check(G, "PDF 164 heading ends 'Bihar'",
      lambda: [l["text"] for l in page(164, CLEAN)["lines"] if "AMANAT" in l["text"]][0].endswith("Bihar"), True)
check(G, "OCR pages unchanged",
      lambda: sum(1 for a, b in zip(PAGES, CLEAN) if a["method"] != "text" and a != b), 0)

# ---------- Unit tests: single functions on tiny inputs ----------
G = "unit"
_n_changes = len(CHANGES)               # unit tests call log(); remove their entries at the end
FN = {1, 2, 3, 13, 19}
for text, want in [
    ("barriers.1 The", "barriers. The"),
    ("Management (PM)2", "Management (PM)"),
    ("Other States13", "Other States"),
    ("Day 1 of training", "Day 1 of training"),
    ("Section 5.3. Process", "Section 5.3. Process"),
    ("COVID-19 response", "COVID-19 response"),
    ("H1N1 virus", "H1N1 virus"),
    ("barriers.5 The", "barriers.5 The"),          # 5 is not a footnote -> keep
]:
    check(G, f"drop_markers: {text!r}", lambda t=text: drop_markers(t, FN), want)

check(G, "heading end marker removed", lambda: drop_heading_end_marker("Programme - Bihar 14", {14}), "Programme - Bihar")
check(G, "heading 'Annexure 2' kept", lambda: drop_heading_end_marker("Annexure 2", {2}), "Annexure 2")

check(G, "footnote: URL continues without space",
      lambda: fix_footnotes(["3 See http://nhm.gov.in/New/Guidance_no", "on_empanelled_20.pdf"], 0)[0][0]["text"],
      "3 See http://nhm.gov.in/New/Guidance_noon_empanelled_20.pdf")
check(G, "footnote: text + URL joined with space",
      lambda: fix_footnotes(["1 Ministry", "www.nhm.gov.in/a.pdf"], 0)[0][0]["text"], "1 Ministry www.nhm.gov.in/a.pdf")
check(G, "footnote: '9' + '9 Complaints' -> one note",
      lambda: fix_footnotes(["9", "9 Complaints pertaining"], 0)[0], [{"num": 9, "text": "9 Complaints pertaining"}])
check(G, "footnote: '*7' number read", lambda: fix_footnotes(["*7 Only suggestive"], 0)[1], {7})
check(G, "footnote: ordinal 'nd' dropped", lambda: fix_footnotes(["nd"], 0)[0], [])

_L = lambda t, lv, sz, top: {"text": t, "level": lv, "font": "x", "size": sz, "top": top}
check(G, "chapter merge (2-line title)",
      lambda: merge_chapter_openers([_L("Chapter", "L1", 14, 60), _L("2", "body", 60, 77),
                                     _L("Human Resource for Health", "L0", 30, 150),
                                     _L("under NHM", "L0", 30, 186)], 0)[0]["text"],
      "Chapter 2: Human Resource for Health under NHM")
check(G, "numbered headings not joined",
      lambda: len(join_split_headings([_L("4.1. A", "L1", 14, 100), _L("4.2. B", "L1", 14, 115)], 0)), 2)
check(G, "normalize_chars", lambda: normalize_chars("“a” – b’s • √"), "\"a\" - b's - Yes")

del CHANGES[_n_changes:]                # keep CHANGES = real run only

# ---------- Report ----------
report = pd.DataFrame(TESTS)
n_pass = int((report["result"] == "PASS").sum())
print(f"{'✅' if n_pass == len(report) else '❌'}  {n_pass}/{len(report)} tests passed\n")
print(report.groupby("group", sort=False)["result"]
            .apply(lambda s: f"{(s == 'PASS').sum()}/{len(s)}").to_string())
report.to_csv(ROOT / "quality" / "test_report.csv", index=False)

failed = report[report["result"] == "FAIL"]
failed if len(failed) else "All tests passed — nothing to show."

✅  47/47 tests passed

group
12 extract    15/15
13 clean      14/14
unit          18/18


'All tests passed — nothing to show.'

## 13.H  Save (only when every test in 13.T passed)

In [ ]:
# 13.H  Save (only when every test in 13.T passed)
n_fail = int((report["result"] == "FAIL").sum())

if n_fail:
    print(f"❌ Not saved: {n_fail} test(s) failing. Fix them and re-run 13.E → 13.T first.")
else:
    # 1. Save the cleaned pages
    with CLEAN_PATH.open("w", encoding="utf-8") as f:
        for p in CLEAN:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

    # 2. Save the change log (what every rule did)
    pd.DataFrame(CHANGES).to_csv(ROOT / "quality" / "step13_changes.csv", index=False)

    # 3. Read the file back and confirm it matches what's in memory
    reloaded = [json.loads(line) for line in CLEAN_PATH.open(encoding="utf-8")]
    same = reloaded == CLEAN

    print(f"✅ Saved {len(reloaded)} pages → {CLEAN_PATH.relative_to(ROOT)}")
    print(f"   Change log: {len(CHANGES)} rows → quality/step13_changes.csv")
    print(f"   Test report → quality/test_report.csv")
    print(f"   Read-back check: {'identical ✅' if same else 'DIFFERENT ❌'}")

✅ Saved 174 pages → data\interim\pages_clean.jsonl
   Change log: 36 rows → quality/step13_changes.csv
   Test report → quality/test_report.csv
   Read-back check: identical ✅


## 14.0 Pre-checking  before start of step 14

In [ ]:
inv = pd.read_csv(PROFILE / "table_inventory.csv")
print(inv.columns.tolist())
inv.head(8)

['page', 'table_no', 'n_rows', 'n_cols', 'fill_percent', 'kind', 'multi_row_header', 'continues_prev', 'top', 'bottom']


,page,table_no,n_rows,n_cols,fill_percent,kind,multi_row_header,continues_prev,top,bottom
0,15,1,19,2,100.00,2-col list,False,False,322.2,760.9
1,16,1,16,2,100.00,2-col list,False,True,54.8,424.6
2,34,1,7,4,75.00,data table,False,False,256.1,386.7
3,52,1,16,2,87.50,data table,True,False,285.8,748.8
4,76,1,4,2,50.00,data table,True,False,163.1,229.9
5,78,1,14,3,90.48,data table,False,False,78.6,522.2
6,82,1,4,4,81.25,data table,False,False,72.6,216.9
7,84,1,5,3,20.00,form grid,False,False,338.4,421.8


In [ ]:
compare(78)

── PDF 78 BEFORE ──
  L1      Induction training Schedule of M&E Officer
  body    Day Thematic area Description
  body    Day 1- Day 8 Initial Orientation
  body    Day 1 Introduction to National Health Objective and strategies, structure and
  body    Mission function of NHM
  body    Day 2 Orientation of Duties and Role of the incumbent, overall
  body    Responsibilities responsibilities specific to monitoring and
  body    evaluation
  body    Day 3 Monitoring and evaluation Constitution of M&E Division, Basic duties
  body    system of each team member. Details of all
  body    process related to Resource allocation and
  body    utilization
  body    Day 4 Introduction to the partner Sessions on the role and involvement
  body    departments of water & sanitation, women & child
  body    development
  [fn] 12 The induction training should preferably be three to five days at a stretch. The comple
  [fn] three phases.
── PDF 78 AFTER ──
  L1      Induction training Schedule of M&E

## 14.0  Set-Up

In [ ]:
import re, json, copy
from collections import Counter
import pandas as pd
import pdfplumber
from src.paths import ROOT, PDF_PATH, PAGES_CLEAN, PAGES_TABLES, TABLES_DIR, TABLE_INV

CLEAN = [json.loads(line) for line in PAGES_CLEAN.open(encoding="utf-8")]
INV   = pd.read_csv(TABLE_INV)
print(len(CLEAN), "pages ·", len(INV), "tables in inventory on", INV["page"].nunique(), "pages")
print(INV["kind"].value_counts().to_string())

FORMS_MODE     = "fields"     # "fields" = keep form field names · "skip" = leave forms out of search
HEADER_ROWS    = {101: 3}     # pages whose header spans N rows (default 2 when multi_row_header)
TABLE_SETTINGS = {}           # ⚠️ copy the settings notebook 01 used for find_tables ({} = default)
HEADING_LEVELS_T = {"L0", "L1", "L2", "L3", "caption"}

def as_bool(v):
    return str(v).strip().lower() in {"true", "1", "yes", "y"}

def norm_kind(k):
    return str(k).strip().lower().replace(" ", "_").replace("-", "_")

174 pages · 36 tables in inventory on 24 pages
kind
data table    19
form grid     15
2-col list     2


## 14.1 Registries

In [ ]:
# A registry maps a NAME to the FUNCTION that handles it.
HEADER_STRATEGIES = {}   # name -> {"when": test(inv_row, prev_table), "build": fn}   first match wins
ROW_TYPES         = {}   # name -> {"when": test(cells, state),       "apply": fn}   first match wins
KIND_HANDLERS     = {}   # kind -> fn(table) -> list of texts
FORM_MODES        = {}   # mode -> fn(table) -> list of texts

def header_strategy(name, when):
    def deco(fn):
        HEADER_STRATEGIES[name] = {"when": when, "build": fn}
        return fn
    return deco

def row_type(name, when):
    def deco(fn):
        ROW_TYPES[name] = {"when": when, "apply": fn}
        return fn
    return deco

def kind_handler(*kinds):
    def deco(fn):
        for k in kinds:
            KIND_HANDLERS[norm_kind(k)] = fn
        return fn
    return deco

def form_mode(name):
    def deco(fn):
        FORM_MODES[name] = fn
        return fn
    return deco

def first_match(registry, *args):
    """(name, entry) of the first registered entry whose `when` test is True — order = registration order."""
    return next((name, e) for name, e in registry.items() if e["when"](*args))

## 14.2 Header strategies (order matters: the fallback goes last)

In [ ]:
def column_names(header):
    return [h if h else f"col{j + 1}" for j, h in enumerate(header)]

@header_strategy("continued", when=lambda inv, prev: as_bool(inv["continues_prev"]) and prev is not None)
def header_continued(rows, inv, prev):
    header = prev["header"]
    body = rows[1:] if rows and rows[0] == header else rows       # header repeated on the new page
    return header, body

@header_strategy("multi_row", when=lambda inv, prev: as_bool(inv["multi_row_header"]) or int(inv["page"]) in HEADER_ROWS)
def header_multi_row(rows, inv, prev):
    k = HEADER_ROWS.get(int(inv["page"]), 2)
    head = [list(r) for r in rows[:k]]
    for r in head:                                    # merged cells come back empty: fill from the left
        for j in range(1, len(r)):
            r[j] = r[j] or r[j - 1]
    header = [" - ".join(dict.fromkeys(r[j] for r in head if r[j])) for j in range(len(rows[0]))]
    return column_names(header), rows[k:]

@header_strategy("simple", when=lambda inv, prev: True)             # fallback
def header_simple(rows, inv, prev):
    return column_names(rows[0]), rows[1:]

## 14.3 Row types for data tables

In [ ]:
def filled(cells):
    return [c for c in cells if c]

@row_type("empty", when=lambda cells, st: not filled(cells))
def row_empty(cells, st):
    pass

@row_type("group", when=lambda cells, st: len(cells) > 1 and cells[0] and not filled(cells[1:]))
def row_group(cells, st):                     # "Day 1- Day 8 Initial Orientation" across the whole row
    st["group"] = cells[0]

@row_type("wrapped", when=lambda cells, st: not cells[0] and st["rows"] and filled(cells[1:]))
def row_wrapped(cells, st):                   # text spilled into a new row: add it to the row above
    prev = st["rows"][-1]["cells"]
    for j, c in enumerate(cells):
        prev[j] = (prev[j] + " " + c).strip() if c else prev[j]

@row_type("normal", when=lambda cells, st: True)                    # fallback
def row_normal(cells, st):
    st["rows"].append({"group": st["group"], "cells": list(cells)})

def row_to_text(row, header):
    pairs = [v if re.fullmatch(r"col\d+", h) else f"{h}: {v}"      # no "col2:" labels in the text
             for h, v in zip(header, row["cells"]) if v]
    return " | ".join(([row["group"]] if row["group"] else []) + pairs)

## 14.4 Kind handlers

In [ ]:
def real_cells(table):
    return [c for r in [table["header"], *table["body"]] for c in r if c and not re.fullmatch(r"col\d+", c)]

@kind_handler("data", "data_table")
def texts_for_data(table):
    st = {"group": None, "rows": []}
    for cells in table["body"]:
        name, entry = first_match(ROW_TYPES, cells, st)
        entry["apply"](cells, st)
    table["rows"] = [r["cells"] for r in st["rows"]]
    return [row_to_text(r, table["header"]) for r in st["rows"]]

@form_mode("fields")
def form_fields(table):
    cells = real_cells(table)
    return ["Form fields: " + ", ".join(dict.fromkeys(cells))] if cells else []

@form_mode("skip")
def form_skip(table):
    return []

@kind_handler("form", "form_grid", "grid")
def texts_for_form(table):
    return FORM_MODES[FORMS_MODE](table)

@kind_handler("list")
def texts_for_list(table):
    return ["- " + c for c in real_cells(table)]

@kind_handler("2_col_list")
def texts_for_two_col_list(table):
    """Each row is a pair -> '- left: right'"""
    items = []
    for r in [table["header"], *table["body"]]:
        cells = [c for c in r if c and not re.fullmatch(r"col\d+", c)]
        if cells:
            items.append("- " + ": ".join(cells))
    return items

def texts_for_header_only(table):
    """Tables with no body rows (text boxes, one-row grids): the cells themselves are the content"""
    cells = real_cells(table)
    return [" | ".join(cells)] if cells else []

@kind_handler("data", "data_table")
def texts_for_data(table):
    st = {"group": None, "rows": []}
    for cells in table["body"]:
        name, entry = first_match(ROW_TYPES, cells, st)
        entry["apply"](cells, st)
    table["rows"] = [r["cells"] for r in st["rows"]]
    return [row_to_text(r, table["header"]) for r in st["rows"]] or texts_for_header_only(table)

def texts_for(table):
    kind = norm_kind(table["kind"])
    assert kind in KIND_HANDLERS, f"No handler for kind {kind!r}. Registered: {sorted(KIND_HANDLERS)}"
    return KIND_HANDLERS[kind](table)



## 14.5 Reading tables from the PDF

In [ ]:
def clean_cell(c):
    return normalize_chars(re.sub(r"\s+", " ", c or "").strip())

def read_tables(pdf_page_obj):
    found = [{"top": t.bbox[1], "bottom": t.bbox[3],
              "raw": [[clean_cell(c) for c in r] for r in t.extract()]}
             for t in pdf_page_obj.find_tables(table_settings=TABLE_SETTINGS)]
    return sorted(found, key=lambda t: t["top"])

def pair_with_inventory(found, inv_rows):
    """Match each inventory row to the detected table whose top is closest."""
    pairs, free = [], list(found)
    for _, inv in inv_rows.sort_values("table_no").iterrows():
        best = min(free, key=lambda t: abs(t["top"] - float(inv["top"])), default=None)
        pairs.append((inv, best))
        free = [t for t in free if t is not best]
    return pairs

def table_title(lines, top, prev):
    above = [l for l in lines if l["top"] < top and l["level"] in HEADING_LEVELS_T]
    return above[-1]["text"] if above else (prev["title"] if prev else None)

## 14.6 Swap the mixed lines for a marker

In [ ]:
def replace_lines(lines, tables):
    out = list(lines)
    for t in tables:
        inside = [l for l in out if t["top"] - 2 <= l["top"] <= t["bottom"] + 2]
        t["removed_chars"] = sum(len(l["text"]) for l in inside)
        out = [l for l in out if l not in inside]
        out.append({"text": f"[table {t['id']}]", "level": "table", "table_id": t["id"],
                    "font": None, "size": None, "top": round(t["top"], 1)})
    return sorted(out, key=lambda l: l["top"])

## 14.7 Run

In [ ]:
def build_tables(clean_pages, inv, pdf_path):
    pages = copy.deepcopy(clean_pages)                 # CLEAN stays untouched for comparison
    by_page = {p["pdf_page"]: p for p in pages}
    for p in pages:
        p["tables"] = []
    prev = None
    with pdfplumber.open(pdf_path) as pdf:
        for pdf_no, inv_rows in inv.groupby("page", sort=True):
            page  = by_page[int(pdf_no)]
            found = read_tables(pdf.pages[int(pdf_no) - 1])
            if len(found) != len(inv_rows):
                print(f"⚠️ PDF {pdf_no}: found {len(found)} tables, inventory says {len(inv_rows)}")
            for inv_row, t in pair_with_inventory(found, inv_rows):
                if t is None:
                    print(f"⚠️ PDF {pdf_no} table {inv_row['table_no']}: not found")
                    continue
                name, strat  = first_match(HEADER_STRATEGIES, inv_row, prev)
                header, body = strat["build"](t["raw"], inv_row, prev)
                table = {"id": f"{int(pdf_no)}-{int(inv_row['table_no'])}", "pdf_page": int(pdf_no),
                         "table_no": int(inv_row["table_no"]), "kind": norm_kind(inv_row["kind"]),
                         "header_strategy": name, "title": table_title(page["lines"], t["top"], prev),
                         "header": header, "body": body, "n_rows_raw": len(t["raw"]),
                         "raw_chars": sum(len(c) for r in t["raw"] for c in r),
                         "top": round(t["top"], 1), "bottom": round(t["bottom"], 1)}
                table["texts"] = texts_for(table)
                table.setdefault("rows", body)
                del table["body"]
                page["tables"].append(table)
                prev = table
            page["lines"] = replace_lines(page["lines"], page["tables"])
    return pages

#KIND_HANDLERS["2_col_list"] = texts_for_list

TPAGES = build_tables(CLEAN, INV, PDF_PATH)
# ALL_T = [t for p in TPAGES for t in p["tables"]]
# print(len(ALL_T), "tables built ·", dict(Counter(t["header_strategy"] for t in ALL_T)))
ALL_T  = [t for p in TPAGES for t in p["tables"]]
print(len(ALL_T), "tables built ·", dict(Counter(t["header_strategy"] for t in ALL_T)))

36 tables built · {'simple': 19, 'continued': 6, 'multi_row': 11}


## 14.8 Look at one page's tables

In [ ]:
def show_table(pdf_no, n=8):
    page = next(p for p in TPAGES if p["pdf_page"] == pdf_no)
    for t in page["tables"]:
        print(f"── table {t['id']} · {t['kind']} · header: {t['header_strategy']} · title: {t['title']}")
        print("   columns:", t["header"])
        for x in t["texts"][:n]:
            print("   →", x[:150])

show_table(78)

── table 78-1 · data_table · header: simple · title: Induction training Schedule of M&E Officer
   columns: ['Day', 'Thematic area', 'Description']
   → Day 1- Day 8 Initial Orientation | Day: Day 1 | Thematic area: Introduction to National Health Mission | Description: Objective and strategies, struct
   → Day 1- Day 8 Initial Orientation | Day: Day 2 | Thematic area: Orientation of Duties and Responsibilities | Description: Role of the incumbent, overal
   → Day 1- Day 8 Initial Orientation | Day: Day 3 | Thematic area: Monitoring and evaluation system | Description: Constitution of M&E Division, Basic dut
   → Day 1- Day 8 Initial Orientation | Day: Day 4 | Thematic area: Introduction to the partner departments | Description: Sessions on the role and involve
   → Day 1- Day 8 Initial Orientation | Day: Day 5 | Thematic area: Administrative Orientation | Description: Reporting details of the various programs und
   → Day 1- Day 8 Initial Orientation | Day: Day 6 | Thematic area: Intr

## 14.T Tests (run after 13.T, which defines check)

In [ ]:
TESTS = [t for t in TESTS if not t["group"].startswith("14")]      # re-running replaces old 14.x results
G = "14 tables"
ALL_T = [t for p in TPAGES for t in p["tables"]]
inv_n_rows = {(int(r.page), int(r.table_no)): int(r.n_rows) for r in INV.itertuples()}

check(G, "tables found = inventory", lambda: len(ALL_T), len(INV))
check(G, "pages with tables = inventory", lambda: len({t["pdf_page"] for t in ALL_T}), INV["page"].nunique())
check(G, "per-page counts match inventory",
      lambda: sorted(pg for pg in set(INV["page"]) | {t["pdf_page"] for t in ALL_T}
                     if sum(t["pdf_page"] == pg for t in ALL_T) != int((INV["page"] == pg).sum())), [])
check(G, "kinds match inventory",
      lambda: dict(sorted(Counter(t["kind"] for t in ALL_T).items())),
      dict(sorted(Counter(norm_kind(k) for k in INV["kind"]).items())))
check(G, "every kind has a handler", lambda: sorted({norm_kind(k) for k in INV["kind"]} - set(KIND_HANDLERS)), [])
check(G, "row counts match inventory",
      lambda: [(t["id"], t["n_rows_raw"], inv_n_rows[(t["pdf_page"], t["table_no"])]) for t in ALL_T
               if t["n_rows_raw"] != inv_n_rows[(t["pdf_page"], t["table_no"])]], [])
check(G, "continued tables use previous header",
      lambda: sorted(f"{int(r.page)}-{int(r.table_no)}" for r in INV.itertuples() if as_bool(r.continues_prev))
              == sorted(t["id"] for t in ALL_T if t["header_strategy"] == "continued"), True)
check(G, "PDF 101 header: no empty names",
      lambda: [h for t in ALL_T if t["pdf_page"] == 101 for h in t["header"] if re.fullmatch(r"col\d+", h)], [])
check(G, "every data table has row texts", lambda: [t["id"] for t in ALL_T if t["kind"].startswith("data") and not t["texts"]], [])
check(G, "no table loses its text",
      lambda: [t["id"] for t in ALL_T if t["removed_chars"] > 0 and not t["texts"]
               and not (FORMS_MODE == "skip" and KIND_HANDLERS[t["kind"]] is texts_for_form)], [])
check(G, "no leftover lines inside table boxes",
      lambda: [(p["pdf_page"], l["text"][:40]) for p in TPAGES for t in p["tables"] for l in p["lines"]
               if l["level"] != "table" and t["top"] - 2 <= l["top"] <= t["bottom"] + 2], [])
check(G, "one marker per table", lambda: sum(l["level"] == "table" for p in TPAGES for l in p["lines"]), len(ALL_T))
check(G, "content kept (table chars / removed chars)",
      lambda: round(sum(t["raw_chars"] for t in ALL_T) / max(1, sum(t["removed_chars"] for t in ALL_T)), 2),
      lambda v: 0.8 <= v <= 1.25, "between 0.8 and 1.25")
check(G, "pages without tables unchanged",
      lambda: [a["pdf_page"] for a, b in zip(CLEAN, TPAGES) if not b["tables"] and a["lines"] != b["lines"]], [])
check(G, "PDF 78: Day 3 row found",
      lambda: any("Day 3" in x and "Monitoring" in x for t in ALL_T if t["pdf_page"] == 78 for x in t["texts"]), True)

G = "14 unit"
_inv = lambda c, m: {"continues_prev": c, "multi_row_header": m, "page": 5}
check(G, "strategy: continued (with previous table)", lambda: first_match(HEADER_STRATEGIES, _inv(True, True), {"header": []})[0], "continued")
check(G, "strategy: continued without previous -> multi_row", lambda: first_match(HEADER_STRATEGIES, _inv(True, True), None)[0], "multi_row")
check(G, "strategy: simple", lambda: first_match(HEADER_STRATEGIES, _inv(False, False), None)[0], "simple")
_st = {"group": None, "rows": [{"group": None, "cells": ["Day 2", "Duties", "Role of"]}]}
check(G, "row type: group",   lambda: first_match(ROW_TYPES, ["Day 1- Day 8 Initial Orientation", "", ""], _st)[0], "group")
check(G, "row type: wrapped", lambda: first_match(ROW_TYPES, ["", "", "the incumbent"], _st)[0], "wrapped")
check(G, "row type: empty",   lambda: first_match(ROW_TYPES, ["", "", ""], _st)[0], "empty")
check(G, "row type: normal",  lambda: first_match(ROW_TYPES, ["Day 3", "M&E", "Duties"], _st)[0], "normal")
check(G, "data texts: group + wrapped",
      lambda: texts_for({"kind": "data", "header": ["Day", "Area"],
                         "body": [["Week 1", ""], ["Day 1", "Intro to"], ["", "NHM"]]}),
      ["Week 1 | Day: Day 1 | Area: Intro to NHM"])
_mode = FORMS_MODE
check(G, "form: field names", lambda: texts_for({"kind": "form", "header": ["Name", "col2"], "body": [["Date of joining", ""]]}),
      ["Form fields: Name, Date of joining"])
FORMS_MODE = "skip"
check(G, "form: skip mode", lambda: texts_for({"kind": "form", "header": ["Name"], "body": []}), [])
FORMS_MODE = _mode
check(G, "list items", lambda: texts_for({"kind": "list", "header": ["A", "col2"], "body": [["B", ""]]}), ["- A", "- B"])

report = pd.DataFrame(TESTS)
n_pass = int((report["result"] == "PASS").sum())
print(f"{'✅' if n_pass == len(report) else '❌'}  {n_pass}/{len(report)} tests passed\n")
print(report.groupby("group", sort=False)["result"].apply(lambda s: f"{(s == 'PASS').sum()}/{len(s)}").to_string())
report.to_csv(ROOT / "quality" / "test_report.csv", index=False)
failed = report[report["result"] == "FAIL"]
failed if len(failed) else "All tests passed — nothing to show."

✅  73/73 tests passed

group
12 extract    15/15
13 clean      14/14
unit          18/18
14 tables     15/15
14 unit       11/11


'All tests passed — nothing to show.'

In [ ]:
for tid in ["101-1", "76-1", "95-1"]:
    t = next(t for t in ALL_T if t["id"] == tid)
    print(f"{tid} · strategy {t['header_strategy']} · header {t['header'][:4]}")
    print("   →", (t["texts"] or ["(none)"])[0][:120])

101-1 · strategy multi_row · header ['Years of Experience', 'Position: Consultant* - Medical Graduate with PG - Entry level Salary', 'Position: Consultant* - Medical Graduate with PG - Salary of existing staff with 5% increment', 'Position: Consultant* - Full-time Masters / PG in Health/ Rural/ Hospital/ Other Management Sciences (AICTE Approved regular course of 2 years) - Entry level Salary']
   → Years of Experience: 0 | Position: Consultant* - Medical Graduate with PG - Entry level Salary: 45000 | Position: Consul
76-1 · strategy multi_row · header ['Basic - HRA', 'Basic - HRA']
   → Basic - HRA | Basic - HRA | Conveyance Allowance | Communication Allowance
95-1 · strategy multi_row · header ['Period of stay - From', 'Period of stay - To', 'Name of the Hotel/ establishment - To', 'Daily rate of boarding/ lodging charged (Inclusive of all taxes) - To']
   → Period of stay - From | Period of stay - To | Name of the Hotel/ establishment - To | Daily rate of boarding/ lodging ch


## 14.H to save pages_tables.jsonl

In [ ]:
# 14.H  Save (only when every test in 14.T passed)
n_fail = int((report["result"] == "FAIL").sum())

if n_fail:
    print(f"❌ Not saved: {n_fail} test(s) failing. Fix them and re-run 14.7 → 14.T first.")
else:
    # 1. Pages with tables → one JSON line per page
    with PAGES_TABLES.open("w", encoding="utf-8") as f:
        for p in TPAGES:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

    # 2. One CSV per table (easy to open in Excel and check by eye)
    TABLES_DIR.mkdir(parents=True, exist_ok=True)
    for t in ALL_T:
        width = len(t["header"])
        rows  = [(list(r) + [""] * width)[:width] for r in t["rows"]]    # every row padded to the header width
        pd.DataFrame(rows, columns=t["header"]).to_csv(TABLES_DIR / f"{t['id']}.csv", index=False)

    # 3. Read the JSONL back and confirm it matches memory
    reloaded = [json.loads(line) for line in PAGES_TABLES.open(encoding="utf-8")]
    n_csv    = len(list(TABLES_DIR.glob("*.csv")))

    print(f"✅ Saved {len(reloaded)} pages → {PAGES_TABLES.relative_to(ROOT)}")
    print(f"   {n_csv} table CSVs → {TABLES_DIR.relative_to(ROOT)}")
    print(f"   Test report → quality/test_report.csv")
    print(f"   Read-back check: {'identical ✅' if reloaded == TPAGES else 'DIFFERENT ❌'}")

✅ Saved 174 pages → data\interim\pages_tables.jsonl
   36 table CSVs → data\interim\tables
   Test report → quality/test_report.csv
   Read-back check: identical ✅
